# W8C1 solutions: from LLM to agent

The complete version of `../exercise/lab.ipynb`, including the post-break `get_today` challenge. Try the lab first; open this when you are stuck on one step.

The one sentence: **an agent is an LLM in a loop, choosing tools and reading their results.**

## Connect

In [ ]:
import json
import math
import requests
from openai import OpenAI

client = OpenAI(
    base_url="https://spark.cs.trinity.edu/v1",
    api_key="sk-trin-...",  # your key
)
MODEL = "qwen2.5-7b"

## The model on its own

It cannot do exact arithmetic, it does not know today's weather, and it forgets you between conversations.

In [ ]:
def chat(prompt):
    resp = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}])
    return resp.choices[0].message.content

print(chat("What is 48291 * 3917?"))
print(48291 * 3917)
print(chat("What is the temperature in San Antonio right now?"))

## Tool 1: a calculator

A tool is two things: **a Python function**, and **a description** the model reads to decide when to call it.

In [ ]:
def calculator(expression):
    return eval(expression, {"__builtins__": {}}, vars(math))

calculator("48291 * 3917")

In [ ]:
def describe(fn, description, **params):
    return {"type": "function", "function": {
        "name": fn.__name__,
        "description": description,
        "parameters": {"type": "object",
                       "properties": {k: {"type": "string", "description": v} for k, v in params.items()},
                       "required": list(params)},
    }}

tools = {"calculator": calculator}
schemas = [describe(calculator, "Evaluate a math expression in Python syntax.", expression="e.g. '2**10 / 7'")]

## The agent loop

The model never runs anything. It *asks* for a tool; **we** run it and hand back the result. Repeat until it answers in plain text.

In [ ]:
def agent(messages):
    while True:
        msg = client.chat.completions.create(model=MODEL, messages=messages, tools=schemas).choices[0].message
        messages.append(msg)
        if not msg.tool_calls:
            return msg.content
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments)
            result = tools[call.function.name](**args)
            print(f"  -> {call.function.name}({args}) = {result}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})

In [ ]:
agent([{"role": "user", "content": "What is 48291 * 3917?"}])

## Tool 2: the weather, from Open-Meteo

No key needed. Two calls: city name to coordinates, then coordinates to the current weather.

In [ ]:
def get_weather(city):
    place = requests.get("https://geocoding-api.open-meteo.com/v1/search", params={"name": city, "count": 1}).json()["results"][0]
    weather = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": place["latitude"], "longitude": place["longitude"],
        "current": "temperature_2m,wind_speed_10m", "temperature_unit": "fahrenheit",
    }).json()["current"]
    return f"{place['name']}: {weather['temperature_2m']}°F, wind {weather['wind_speed_10m']} km/h"

get_weather("San Antonio")

In [ ]:
tools["get_weather"] = get_weather
schemas.append(describe(get_weather, "Get the current weather for a city.", city="city name, e.g. 'Austin'"))

agent([{"role": "user", "content": "Is it warmer in San Antonio or in Chicago right now, and by how many degrees?"}])

## Tool 3: memory, as a JSON file

The conversation ends, the file stays. That is all long-term memory is.

In [ ]:
def remember(fact):
    try:
        facts = json.load(open("memory.json"))
    except FileNotFoundError:
        facts = []
    facts.append(fact)
    json.dump(facts, open("memory.json", "w"))
    return "saved"

def recall():
    try:
        return json.load(open("memory.json"))
    except FileNotFoundError:
        return []

In [ ]:
tools["remember"] = remember
tools["recall"] = recall
schemas.append(describe(remember, "Save a fact about the user for future conversations.", fact="one short fact"))
schemas.append(describe(recall, "Read every saved fact about the user."))

agent([{"role": "user", "content": "Hi! My name is Ed and I live in San Antonio. Please remember that."}])

## A brand-new conversation

A fresh message list knows nothing, not even that a memory exists. One system message tells it to look, and it chains `recall` into `get_weather`.

In [ ]:
agent([
    {"role": "system", "content": "You have long-term memory about the user. Before answering anything that depends on who the user is or where they are, call recall."},
    {"role": "user", "content": "What's the weather where I live, in Celsius?"},
])

## Post-break challenge: what day is it?

Write a `get_today` tool from scratch, then ask the agent the two questions below and check its answers against the plain-Python line under each.

The model does not know today's date, so ask it first. A tool with no inputs is the same shape as `recall`.

In [ ]:
print(chat("How many days until Thanksgiving?"))

In [ ]:
from datetime import date, timedelta

def get_today():
    return date.today().strftime("%A, %B %d, %Y")

tools["get_today"] = get_today
schemas.append(describe(get_today, "Get today's date and day of the week."))
get_today()

In [ ]:
print(agent([{"role": "user", "content": "How many days until Thanksgiving?"}]))
print((date(2026, 11, 26) - date.today()).days)

In [ ]:
print(agent([{"role": "user", "content": "What day of the week will it be 100 days from now?"}]))
print((date.today() + timedelta(days=100)).strftime("%A"))